# Task 5: Speaker X Retrieval and Timestamps

Use the specified `pyannote/embedding` model and supplied reference voice. All YCSEP channels are searched. This is separate from the ASR train/test split. **Execution, calibration and final exports are not yet complete.**

Assumption: annotated speaker IDs are consistent within each video, not globally. Up to three separated representative clips per speaker/video group limit compute. This can miss a speaker if annotations are inconsistent or representative windows contain overlap; the final review must inspect candidate and near-threshold groups.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'speaker-detection':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'speaker-detection'))
CSV = Path(os.environ.get('YCSEP_CSV', ROOT / 'test_docs/test/runtime/local-prep-0316/data/YCSEP_static.csv'))
REFERENCE = Path(os.environ.get('SPEAKER_REFERENCE', ROOT / 'test_docs/test/runtime/speaker-reference'))
OUTPUT = Path(os.environ.get('SPEAKER_OUTPUT', ROOT / 'test_docs/test/runtime/speaker-full'))
DEVICE = os.environ.get('SPEAKER_DEVICE', 'cuda')


## Reference Prototype
Audio is decoded to mono 16 kHz. Nonquiet five-second windows produce L2-normalized 512-dimensional embeddings; their normalized mean is the reference prototype. Reference-window agreement is only a sanity check, not cross-recording threshold calibration. Model and reference hashes are retained. Set `HF_TOKEN` securely or use a cached authorized model; never print the token.

In [ ]:
from prepare_reference import prepare
if not (REFERENCE / 'reference-audit.json').exists():
    prepare(REFERENCE, device=DEVICE)
reference_audit = json.loads((REFERENCE / 'reference-audit.json').read_text())
reference_audit


## Corpus Retrieval
Fetch exact PCM ranges from original YCSEP WAV objects during the clip-service outage. Prefer two-to-fifteen-second annotated regions, centrally capped at five seconds for embedding. Temporal separation reduces dependence on one noisy window. Median similarity across available representatives is the group score. Failed clips remain explicit and are retried without recomputing successful embeddings. Original annotations are retained for localization; their boundaries are not independently verified by this retrieval stage.

In [ ]:
from argparse import Namespace
from retrieve_speakers import run
report_path = OUTPUT / 'retrieval-result.json'
if not report_path.exists() or json.loads(report_path.read_text()).get('failed_groups'):
    run(Namespace(csv=CSV, reference=REFERENCE / 'reference-embeddings.npz', output=OUTPUT,
                  workers=16, device=DEVICE, limit_groups=0))
retrieval_report = json.loads(report_path.read_text())
retrieval_report


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
scores = pd.read_json(OUTPUT / 'group-scores.jsonl', lines=True)
scores['median_similarity'].dropna().hist(bins=60, figsize=(8, 3), color='#477c91')
plt.xlabel('Cosine similarity to Speaker X reference')
plt.ylabel('Speaker/video groups')
plt.show()
scores[['channel', 'file', 'speaker', 'median_similarity', 'failed_rows']].head(25)


## Calibration and Localization Review
Before final export, label cross-recording candidate and noncandidate clips by comparison with the reference, including near-threshold examples and different channels. Select a threshold from those labels, report precision/recall on separate reviewed examples, and show sensitivity to threshold changes. Do not invent labels or interpret a histogram gap as measured accuracy.

Review the selected groups' individual annotated segments for label drift, overlap and boundary errors. Preserve Speaker X speech during overlap. Export sorted nonoverlapping intervals per actual YouTube title as `detected_timestamps.json` and the matching unique titles as `detected_titles.txt`. Disclose any annotation-based boundary assumption and unresolved retrieval failures.

Pending: reviewed calibration labels, threshold decision, localization checks and final export cells/results. This notebook is currently an executable retrieval workflow, not the final Task 5 deliverable.